In [ ]:
import os
import sys

from pyspark.sql import SparkSession
from delta import configure_spark_with_delta_pip

from pyspark.sql.functions import *
from delta.tables import DeltaTable

# Use the Python executable from the active environment
python_path = sys.executable

os.environ["PYSPARK_PYTHON"] = python_path
os.environ["PYSPARK_DRIVER_PYTHON"] = python_path

# Prefer local loopback for Spark networking
os.environ["SPARK_LOCAL_IP"] = "127.0.0.1"

builder = SparkSession.builder \
    .appName("PySparkDemo") \
    .master("local[*]") \
    .config("spark.pyspark.python", python_path) \
    .config("spark.pyspark.driver.python", python_path) \
    .config("spark.driver.host", "127.0.0.1") \
    .config("spark.driver.bindAddress", "127.0.0.1") \
    .config(
        "spark.sql.extensions",
        "io.delta.sql.DeltaSparkSessionExtension"
    ) \
    .config(
        "spark.sql.catalog.spark_catalog",
        "org.apache.spark.sql.delta.catalog.DeltaCatalog"
    )
spark = configure_spark_with_delta_pip(builder).getOrCreate()
print("SparkSession created successfully.")

In [ ]:
optimize_path = "output/delta/optimization_demo"

large_df = spark.range(0, 100000) \
    .withColumnRenamed("id", "OrderID") \
    .withColumn(
        "CustomerID",
        col("OrderID") % 1000
    ) \
    .withColumn(
        "Amount",
        (col("OrderID") % 5000) + 100
    )

large_df.repartition(40) \
    .write \
    .format("delta") \
    .mode("overwrite") \
    .save(optimize_path)

In [ ]:
large_df.show(10)

In [ ]:
large_df.printSchema()

print("Total Records:", large_df.count())

print(
    "Unique Customers:",
    large_df.select("CustomerID").distinct().count()
)

In [ ]:
partitioned_df = large_df.repartition(40)

print(
    "Number of Partitions:",
    partitioned_df.rdd.getNumPartitions()
)

In [ ]:
partitioned_df.write \
    .format("delta") \
    .mode("overwrite") \
    .save(optimize_path)

In [ ]:
for file in os.listdir(optimize_path):
    print(file)

In [ ]:
delta_table = DeltaTable.forPath(
    spark,
    optimize_path
)

In [ ]:
delta_table.detail().show(truncate=False)

In [ ]:
delta_table.detail().select(
    "format",
    "numFiles",
    "sizeInBytes",
    "partitionColumns"
).show(truncate=False)

In [ ]:
import builtins

before_metrics = delta_table.detail() \
    .select("numFiles", "sizeInBytes") \
    .first()

before_files = before_metrics["numFiles"]
before_size = before_metrics["sizeInBytes"]

print("Files Before OPTIMIZE:", before_files)

print(
    "Size Before OPTIMIZE:",
    builtins.round(before_size / 1024, 2),
    "KB"
)

print(
    "Average File Size:",
    builtins.round(before_size / before_files / 1024, 2),
    "KB"
)

In [ ]:
delta_table.optimize().executeCompaction().show(truncate=False)

In [ ]:
delta_table.detail() \
    .select("numFiles", "sizeInBytes") \
    .show()
print("Files Before OPTIMIZE:", before_files)

print(
    "Size Before OPTIMIZE:",
    builtins.round(before_size / 1024, 2),
    "KB"
)

print(
    "Average File Size:",
    builtins.round(before_size / before_files / 1024, 2),
    "KB"
)